## Overview

This analyzes 2023 FAERS adverse event data to provide insights on drug safety and patient risk.

The analysis focuses on known drugs, common adverse reactions, patient demographics, and reporting trends.  

Key outputs include:
- The top 5 drugs with the most death cases  
- The top 5 most frequent adverse reactions  
- Total reported cases by age group and gender  

# Bronze Layer – Load Raw FAERS Data

In [0]:
spark.sql("""
SHOW TABLES IN john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023
""").show(truncate=False)

In [0]:
# Load each table from the catalog

demo = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023.fda_adverse_events_reporting_system_demographics_2023"
)

drug = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023.fda_adverse_events_reporting_system_drug_2023"
)

reaction = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023.fda_adverse_events_reporting_system_drug_reaction_2023"
)

outcome = spark.table(
    "john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023.fda_adverse_events_reporting_system_patient_outcome_2023"
)

# Look at the first 5 rows of each table
demo.show(5)
drug.show(5)
reaction.show(5)
outcome.show(5)

# Silver Layer – Clean and Prepare Data

In [0]:
from pyspark.sql.functions import upper, trim, col

## Clean Drug Name

In [0]:
# Remove extra spaces and convert to uppercase

drug = drug.withColumn(
    "Drug_Name_Clean",
    upper(trim(col("Drug_Name")))
)

## Rename Year and Quarter Columns

In [0]:
# Rename columns in each table so they don't conflict after joins

demo = demo.withColumnRenamed("Year", "Demo_Year")
demo = demo.withColumnRenamed("Quarter", "Demo_Quarter")

drug = drug.withColumnRenamed("Year", "Drug_Year")
drug = drug.withColumnRenamed("Quarter", "Drug_Quarter")

reaction = reaction.withColumnRenamed("Year", "Reaction_Year")
reaction = reaction.withColumnRenamed("Quarter", "Reaction_Quarter")

outcome = outcome.withColumnRenamed("Year", "Outcome_Year")
outcome = outcome.withColumnRenamed("Quarter", "Outcome_Quarter")

## Join All Tables Together (Create Silver Table)

In [0]:
# Join all tables using Primary_Id and Case_Id

silver_df = demo.join(drug, ["Primary_Id", "Case_Id"], "left")
silver_df = silver_df.join(reaction, ["Primary_Id", "Case_Id"], "left")
silver_df = silver_df.join(outcome, ["Primary_Id", "Case_Id"], "left")

## Standardize Patient Outcome

In [0]:
# Make Patient_Outcome uppercase and remove spaces

silver_df = silver_df.withColumn(
    "Patient_Outcome_Normalized",
    upper(trim(col("Patient_Outcome")))
)

## Replace Missing Drug Names

In [0]:
from pyspark.sql.functions import when

silver_df = silver_df.withColumn(
    "Drug_Name_Clean",
    when(col("Drug_Name_Clean").isNull(), "UNKNOWN DRUG")
    .otherwise(col("Drug_Name_Clean"))
)

## Create Readable Age Groups

In [0]:
silver_df = silver_df.withColumn(
    "Age_Group_Readable",
    when(col("Age_Group") == "A", "Adult (18-44)")
    .when(col("Age_Group") == "C", "Child (0-11)")
    .when(col("Age_Group") == "E", "Elderly (65+)")
    .when(col("Age_Group") == "I", "Infant (0-1)")
    .when(col("Age_Group") == "N", "Neonate (0-28 days)")
    .when(col("Age_Group") == "T", "Teen (12-17)")
    .otherwise("Unknown")
)

## Save Silver Table

In [0]:
# Drop table if it exists
spark.sql("DROP TABLE IF EXISTS silver.faers_events")

# Save as Delta table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.faers_events")

# Gold Layer – Analysis Tables

## Top 5 Drugs with Most Deaths

In [0]:
from pyspark.sql.functions import count, sum, col

# Aggregate death counts by drug
gold_death = silver_df \
    .filter(col("Drug_Name_Clean") != "UNKNOWN DRUG") \
    .groupBy("Drug_Name_Clean") \
    .agg(
        count("*").alias("total_reports"),
        sum((col("Patient_Outcome_Normalized") == "DE").cast("int")).alias("death_cases")
    )

# Sort by death count (highest first) and keep top 5
top_5_deaths = gold_death \
    .orderBy(col("death_cases").desc()) \
    .limit(5)

# Save table for dashboard use
top_5_deaths.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top_5_drugs_death")

# Display result
top_5_deaths.show()

## Top 5 Drug Reactions

In [0]:
from pyspark.sql.functions import countDistinct, col

# Count how many cases each reaction appears in
top_5_reactions = silver_df \
    .filter(col("Preferred_Term_For_Event").isNotNull()) \
    .groupBy("Preferred_Term_For_Event") \
    .agg(countDistinct("Case_Id").alias("reaction_count")) \
    .orderBy(col("reaction_count").desc()) \
    .limit(5)

# Save for dashboard use
top_5_reactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top_5_reactions")

# Display result
top_5_reactions.show()

## Total Number of Cases by Age Group

In [0]:
from pyspark.sql.functions import countDistinct

reports_by_age = silver_df \
    .groupBy("Age_Group_Readable") \
    .agg(
        countDistinct("Case_Id").alias("total_reports")
    ) \
    .orderBy("Age_Group_Readable")

# Save for dashboard use
reports_by_age.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.reports_by_age_group")

# Display result
reports_by_age.show()

## Total Number of Cases by Gender

In [0]:
from pyspark.sql.functions import countDistinct, col

cases_by_gender = silver_df \
    .filter(col("Gender_of_Patient").isNotNull()) \
    .groupBy("Gender_of_Patient") \
    .agg(
        countDistinct("Case_Id").alias("total_cases")
    ) \
    .orderBy(col("total_cases").desc())

# Save for dashboard use
cases_by_gender.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.cases_by_gender")

# Display result
cases_by_gender.show()